# Ingestão e análise de segmentação RFM - clientes  de e-commerce

O objetivo desse mini-projeto é aplicar os conceitos de **Engenharia e Análise de Dados** para estudo e construção de portfólio.\
Abaixo um pipeline de dados (camadas **bronze → silver → gold**) foi montado completamente usando os recursos do Databricks com scripts em **Python, PySpark e SQL**.\
\
**Dataset utilizado**: Kaggle -  [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

### Arquitetura do pipeline

Arquivos CSV originais (Kaggle, no volume `workspace.olist.raw`) → camadas **bronze → silver → gold** no catálogo `workspace.olist`:

```
 RAW — arquivos CSV do Kaggle       BRONZE              SILVER                    GOLD
 /Volumes/workspace/olist/raw       workspace.olist     workspace.olist           workspace.olist
 ─────────────────────────────────────────────────────────────────────────────────────────────────────

 olist_orders_dataset.csv         ──► bronze_orders     ┐
 olist_customers_dataset.csv      ──► bronze_customers  ┼──►  silver_pedidos ──┬──►  gold_rfm
 olist_order_payments_dataset.csv ──► bronze_payments   ┘                      ├──►  gold_clientes_resumo
                                                                               ├──►  gold_clientes_uf
                                                                               └──►  dq_resultados
                                                                                     (validações de data quality)
```

In [0]:
# Importando as bibliotecas necessárias
from pyspark.sql import functions as F, Window

# Path dos arquivos originais baixados do Kaggle
base = "/Volumes/workspace/olist/raw"

# Função para carregar os arquivos originais e gravar na camada bronze
def bronze(arquivo, tabela):
    (spark.read.option("header", True).csv(f"{base}/{arquivo}")
        .withColumn("_ingestao_em", F.current_timestamp())
        .write.mode("overwrite").saveAsTable(f"workspace.olist.{tabela}"))

bronze("olist_orders_dataset.csv", "bronze_orders")
bronze("olist_customers_dataset.csv", "bronze_customers")
bronze("olist_order_payments_dataset.csv", "bronze_payments")

In [0]:
# Dataframe de pedidos, removendo pedidos duplicados e filtrando apenas status "delivered"
orders = (spark.table("workspace.olist.bronze_orders")
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .dropDuplicates(["order_id"])
    .filter(F.col("order_status") == "delivered"))

# Dataframe de pagamentos, formatando e filtrando valores negativos
payments = (spark.table("workspace.olist.bronze_payments")
    .withColumn("payment_value", F.col("payment_value").cast("double"))
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .filter(F.col("payment_value") > 0)
    .groupBy("order_id")
    .agg(F.sum("payment_value").alias("valor_pedido"),
         F.max("payment_installments").alias("parcelas"),
         F.max(F.when(F.col("payment_type") == "credit_card", 1).otherwise(0)).alias("usou_cartao")))

# Dataframe de clientes
customers = spark.table("workspace.olist.bronze_customers").select("customer_id", "customer_unique_id", "customer_state")

# Criação da tabela silver_pedidos, agregrando os dataframes acima através das chaves "customer_id" e "order_id"
# IMPORTANTE: "customer_id" parece ser gerado por pedido, então quem identifica o cliente é o campo "customer_unique_id"
silver = (orders.join(customers, "customer_id").join(payments, "order_id")
    .select("order_id", "customer_unique_id", "customer_state",
            "order_purchase_timestamp", "valor_pedido", "parcelas", "usou_cartao"))
silver.write.mode("overwrite").saveAsTable("workspace.olist.silver_pedidos")

# Validações de data quality, registradas na tabela dq_resultados
total = silver.count()
checks = [
    ("order_id duplicado", total - silver.select("order_id").distinct().count()),
    ("valor_pedido nulo", silver.filter(F.col("valor_pedido").isNull()).count()),
    ("data de compra nula", silver.filter(F.col("order_purchase_timestamp").isNull()).count()),
]
(spark.createDataFrame(checks, ["regra", "falhas"])
    .withColumn("executado_em", F.current_timestamp())
    .write.mode("append").saveAsTable("workspace.olist.dq_resultados"))

In [0]:
# Describe da tabela silver_pedidos para visão geral dos dados
(spark.table("workspace.olist.silver_pedidos")
    .describe()
    .display())

summary,order_id,customer_unique_id,customer_state,valor_pedido,parcelas,usou_cartao
count,96477,96477,96477,96477,96477,96477
mean,null,null,null,159.85635716283707,2.928024295946184,0.7701732019030443
stddev,null,null,null,218.81314427999968,2.7127234533320466,0.42072351454773393
min,00010242fe8c5a6d1ba2dd792cb16214,0000366f3b9a7992bf8c76cfdf3221e2,AC,9.59,0,0
max,fffe41c64501cc87c801fd61db3f6244,ffffd2657e2aad2907e67c3e9daecbeb,TO,13664.08,24,1


In [0]:
# Contagem de pedidos por cliente (customer_unique_id) na camada silver
# Apenas 2801 clientes compraram mais de uma vez - cerca de 3% dos 96.477 clientes

(spark.table("workspace.olist.silver_pedidos")
    .groupBy("customer_unique_id")
    .agg(F.count("order_id").alias("qtd_pedidos"))
    .orderBy(F.col("qtd_pedidos").desc())
    .display())

customer_unique_id,qtd_pedidos
8d50f5eadf50201ccdcedfb9e2ac8455,15
3e43e6105506432c953e165fb2acf44c,9
1b6c7548a2a1f9037c1fd3ddfed95f33,7
ca77025e7201e3b30c44b472ff346268,7
6469f99c1f9dfae7733b25662e7f1782,7
dc813062e0fc23409cd255f7f53c7074,6
12f5d6e1cbf93dafd9dcc19095df0b3d,6
f0e310a6839dce9de1638e0fe5ab282a,6
63cfc61cee11cbe306bff5857d00bfe4,6
47c1a3033b8b77b3ab6e109eb4d5fdf3,6


In [0]:
# Usa a última data do dataset como referência, já que os dados originais são entre 2016-2018
ref = silver.agg(F.max("order_purchase_timestamp")).first()[0]

# Agregação dos clientes por RFM
rfm = (silver.groupBy("customer_unique_id")
    .agg(F.datediff(F.lit(ref), F.max("order_purchase_timestamp")).alias("recencia"),
         F.countDistinct("order_id").alias("frequencia"),
         F.sum("valor_pedido").alias("monetario"),
         F.avg("parcelas").alias("parcelas_medias"),
         F.max("usou_cartao").alias("usou_cartao")))

# Ordenação e divisão dos clientes em 5 grupos
scored = (rfm
    .withColumn("r_score", F.ntile(5).over(Window.orderBy(F.col("recencia").desc()))) # maior recência fica no grupo 5 = melhor
    .withColumn("m_score", F.ntile(5).over(Window.orderBy("monetario")))) # gasto monetário: grupo 1 quem gasta menos, grupo 5 quem gasta mais

# OBS: como 97% dos clientes tem apenas 1 pedido, não há segmentação por frequencia

# Regras de segmento e sugestão de leitura dentro do negócio
gold = scored.withColumn("segmento",
    F.when((F.col("frequencia") > 1) & (F.col("r_score") >= 4), "Recorrentes ativos") # comprou mais de uma vez, recência alta - é um cliente fiel
     .when((F.col("r_score") >= 4) & (F.col("m_score") >= 4), "Novos de alto valor") # recência e valor altos, porém comprou apenas 1 vez - incentivar recompra
     .when((F.col("r_score") <= 2) & (F.col("m_score") >= 4), "Alto valor em risco") # alto valor, porém recência baixa - cliente a ser reativado
     .when(F.col("r_score") <= 2, "Inativos") # recência e valor baixos - clientes de baixa prioridade
     .otherwise("Base regular")) # clientes intermediários

gold.write.mode("overwrite").saveAsTable("workspace.olist.gold_rfm")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Visão geral dos clientes
clientes_resumo = (silver.agg(
        F.countDistinct("customer_unique_id").alias("qde_clientes"),
        F.count("order_id").alias("qde_pedidos"),
        F.sum("valor_pedido").alias("faturamento"),
        F.avg("valor_pedido").alias("valor_medio"),
        F.avg("parcelas").alias("parcelas_medias"),
        F.sum("usou_cartao").alias("qde_usou_cartao")))

# Agregação dos clientes por UF
clientes_uf = (silver.groupBy("customer_state")
    .agg(
        F.countDistinct("customer_unique_id").alias("qde_clientes"),
        F.count("order_id").alias("qde_pedidos"),
        F.sum("valor_pedido").alias("faturamento"),
        F.avg("valor_pedido").alias("valor_medio"),
        F.avg("parcelas").alias("parcelas_medias"),
        F.sum("usou_cartao").alias("qde_usou_cartao")))

clientes_resumo.write.mode("overwrite").saveAsTable("workspace.olist.gold_clientes_resumo")
clientes_uf.write.mode("overwrite").saveAsTable("workspace.olist.gold_clientes_uf")